# Spotify Web API — Data Inventory

A complete, honest inventory of every table this project can reach through the Spotify Web API, with a real, redacted sample of each, so the breadth of the data — and its holes — are visible in one document. Surfaces the warehouse already holds are read from it; the rest are called live, once.

Two constraints shape everything below. **This is a development-mode app:** at most five Spotify accounts can authorize it, and the app owner needs Premium. And **it was registered after 2024-11-27**, so audio features, audio analysis, recommendations and related artists are closed to it (§6.1). Profile, user and owner fields are redacted throughout.

Everything starts from the project's notebook setup: it resolves the session, reads credentials from outside the repository, opens the warehouse connection and configures the plotting libraries. The run details and a coverage summary appear below.

In [ ]:
from spotify_lakehouse import inventory
from spotify_lakehouse.notebook import setup

ctx = setup(profile="marc")

The warehouse is read first. Each surface not yet extracted is then called once, paced, while holding the `spot_refresh` lock, so the scheduled poller skips rather than competing for the rate limit.

In [ ]:
captures = inventory.collect(ctx)
inventory.emit_intro(ctx, captures)

## 1. Identity & Account

Who the authorized account is. The warehouse already stores `/me` responses, so this section reads the latest one rather than calling the API.

In [ ]:
inventory.emit_domain(captures, "1")

## 2. Library

Saved tracks, saved albums and followed artists. None is extracted yet, so each is called live once.

In [ ]:
inventory.emit_domain(captures, "2")

## 3. Playlists

The playlist inventory, then the contents of the first playlist. Not extracted yet; called live.

In [ ]:
inventory.emit_domain(captures, "3")

## 4. Listening

Recent plays come from the warehouse, where the poller lands them every 30 minutes. Spotify's own top artists and top tracks are called live for each of their three time ranges.

In [ ]:
inventory.emit_domain(captures, "4")

### 4.4 Local date versus UTC date

Every play is stored with a UTC timestamp, but a day of listening belongs to the listener's own calendar. This shows how many plays would move to a different date if they were grouped by UTC.

The chart counts the same plays twice: once by their UTC date, once by their local date.

<!-- caption: Plays per calendar date, counted by UTC date and by the profile's home-timezone date -->

In [ ]:
inventory.emit_local_vs_utc(ctx)

### 4.5 The 50-item window is the whole API history

How much listening history the API path holds, when it starts, and what has to come from the export.

In [ ]:
inventory.emit_api_window(ctx)

## 5. Catalog Enrichment

Catalog objects that describe what was played. Artists come from the warehouse; albums, tracks, podcasts and search are called live.

In [ ]:
inventory.emit_domain(captures, "5")

## 6. What We Cannot Get

The negative space: what this project cannot obtain from the API, with the round that measured each item. It is here so no future session spends an afternoon looking for something that is gone.

In [ ]:
inventory.emit_cannot_get(ctx, captures)

## 7. Coverage Summary

One row per endpoint section: where its data came from, whether it was reachable, what was sampled, and which warehouse table it feeds.

In [ ]:
inventory.emit_coverage(captures)

The warehouse connection is closed at the end of the run.

In [ ]:
ctx.close()